
# Classes, Inheritance, and Why They Matter for Deep Learning

Notebook by Debbie Pelacani Cruz


## 1. Plain Python: a base `Neuron`

In deep learning, the fundamental computational unit is the neuron. A neuron is characterised by a set of weights and a bias term, and it processes its inputs by computing a weighted sum of the inputs and adding the bias (a linear model).

Let’s make a simple class that represents a neuron.

It should have two variables that define it, the `weights` and `biases`, which we store as class **attributes**. 

The class should also include a function that applies these stored variables to process an input, typically the data. We will call this function `forward`. By defining it with `self`, it becomes a **method** of the class, giving each instance its own behaviour.

In [1]:
# Base class: Neuron
class Neuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def forward(self, inputs):
        # simple weighted sum
        total = sum(w * i for w, i in zip(self.weights, inputs)) + self.bias
        return total

# Try it
n1 = Neuron([0.5, -0.2], 0.1)
print("Neuron 1")
print("Neuron weights:", n1.weights)
print("Neuron bias:", n1.bias)
print("Base neuron output:", n1.forward([4, 3]))

n2 = Neuron([1.0, 1.0], -1.0)
print("\nNeuron 2")
print("Neuron weights:", n2.weights)
print("Neuron bias:", n2.bias)
print("Base neuron output:", n2.forward([4, 3]))


Neuron 1
Neuron weights: [0.5, -0.2]
Neuron bias: 0.1
Base neuron output: 1.5

Neuron 2
Neuron weights: [1.0, 1.0]
Neuron bias: -1.0
Base neuron output: 6.0



## 2. Inheritance: special kinds of neurons

In neural networks, **activation functions** change how neurons behave.

By itself, the neuron we created can only combine inputs as a linear model

But by applying a non-linear function to the neuron’s output, activation functions allow neurons to capture more complex, non-linear patterns in the data.

We could then write:

```python
class ReLUNeuron():
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def relu(self, inputs)
        return max(0, inputs) # a non-linear piecewise function

    def forward(self, inputs):
        total = sum(w * i for w, i in zip(self.weights, inputs)) + self.bias
        return self.relu(0, total)    
```

But rather than rewriting the entire neuron each time, we can define new classes that **inherit** from the base `Neuron` class and modify only the activation behaviour.

We first call the linear model from the parent implementation using `super().forward(...)` and then apply the activation.


In [2]:
import math

# Derived class: ReLU neuron
class ReLUNeuron(Neuron):
    def relu(self, x):
        return max(0, x) # a non-linear piecewise function
    
    def forward(self, inputs):
        total = super().forward(inputs)      # use parent's forward()
        return self.relu(total)              # apply ReLU activation

# Derived class: Sigmoid neuron
class SigmoidNeuron(Neuron):
    def sigmoid(self, x):
        return 1 / (1 + math.exp(-x))  # a non-linear S-shaped function
    
    def forward(self, inputs):
        total = super().forward(inputs)
        return self.sigmoid(total)  # apply sigmoid activation

# Compare outputs
inputs = [2, 3]
base = Neuron([0.5, -0.2], 0.1)
relu = ReLUNeuron([0.5, -0.2], 0.1)
sigmoid = SigmoidNeuron([0.5, -0.2], 0.1)

print("Base:   ", base.forward(inputs))
print("ReLU:   ", relu.forward(inputs))
print("Sigmoid:", sigmoid.forward(inputs))
 


Base:    0.4999999999999999
ReLU:    0.4999999999999999
Sigmoid: 0.6224593312018545


And a few checks:

In [3]:
# Check weights and biases are inherited
print("\nReLUNeuron weights:", relu.weights)
print("ReLUNeuron bias:", relu.bias)
print("SigmoidNeuron weights:", sigmoid.weights)
print("SigmoidNeuron bias:", sigmoid.bias)

# Check the class hierarchy
print("\nClass hierarchy:")
print("ReLUNeuron is subclass of Neuron:", issubclass(ReLUNeuron, Neuron))
print("SigmoidNeuron is subclass of Neuron:", issubclass(SigmoidNeuron, Neuron))
print("ReLUNeuron is subclass of SigmoidNeuron:", issubclass(ReLUNeuron, SigmoidNeuron))   


ReLUNeuron weights: [0.5, -0.2]
ReLUNeuron bias: 0.1
SigmoidNeuron weights: [0.5, -0.2]
SigmoidNeuron bias: 0.1

Class hierarchy:
ReLUNeuron is subclass of Neuron: True
SigmoidNeuron is subclass of Neuron: True
ReLUNeuron is subclass of SigmoidNeuron: False



## 3. From plain Python to PyTorch

Now, modern deep learning models can have over billions of such neurons, each with their own complex behaviours.

`PyTorch` is one of the libraries that provide a highly optimized framework to handle this complexity efficiently, and is built around the same principles of classes and inheritance.

In PyTorch, everything "neural" inherits from `torch.nn.Module`. This base class manages parameters, device movement, saving/loading, and integrates with the autograd engine.


In [4]:
# If you want to run this yourselves now, you will need to "pip install torch" first
import torch
import torch.nn as nn
import torch.nn.functional as F
TORCH_AVAILABLE = True
print("PyTorch imported successfully.")

# set seed
torch.manual_seed(42)

PyTorch imported successfully.



### 3.1 A custom PyTorch neuron

We inherit from `nn.Module`, define a small linear transform, and apply ReLU in `forward`.  
Key points:
- `super().__init__()` initialises `nn.Module`.
- Layers like `nn.Linear` register learnable parameters automatically.
- `forward` defines the computation.


In [5]:
class CustomNeuron(nn.Module):
    def __init__(self, input_size):
        super().__init__()                      # call parent nn.Module constructor
        self.linear = nn.Linear(input_size, 1)  # built-in linear layer (weights and biases are handled internally)
        self.relu = nn.ReLU()                   # built-in ReLU activation, a class also inheriting from nn.Module

    def forward(self, x):
        z = self.linear(x)      # weighted sum + bias -- equivalent to calling Neuron.forward()
        return self.relu(z)     # apply ReLU -- equivalent to ReLUNeuron.forward()

# Test run
neuron = CustomNeuron(3)  # 3 input feature
x = torch.tensor([[1.0, 2.0, 3.0],
                    [4.0, 5.0, 6.0]])  # batch of 2
outuput = neuron(x) # equivalent to neuron.forward(x)
print("CustomNeuron output:\n", outuput)



CustomNeuron output:
 tensor([[1.5244],
        [3.8805]], grad_fn=<ReluBackward0>)



### 3.2 From neurons to networks

Here is a tiny multi-layer perceptron (MLP) that also inherits from `nn.Module`.  
It stacks layers and uses `forward` to define the data flow.


In [16]:

class TinyMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate and run a forward pass
mlp = TinyMLP(input_size=3, hidden_size=4, num_classes=2)
x = torch.randn(5, 3)  # batch of 5
logits = mlp(x)
print("TinyMLP output shape:", logits.shape)  # (5, 2)


TinyMLP output shape: torch.Size([5, 2])


We can then use it to train these models:

```python
# Training loop example
    torch.manual_seed(0)
    model = TinyMLP(input_size=3, hidden_size=8, num_classes=2)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()

    # Synthetic data (10 samples, 3 features), random binary labels
    X = torch.randn(10, 3)
    y = torch.randint(0, 2, (10,))

    for step in range(50):
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        if (step + 1) % 10 == 0:
            print(f"Step {step+1:02d} | Loss: {loss.item():.4f}")
```

But the excitement part will come later...

## Key Takeways
- **Classes** encapsulate data (attributes) and behaviour (methods).  
- **Inheritance** lets you define a general blueprint once and specialise behaviour in subclasses.  
- **PyTorch** uses this pattern pervasively: models and layers **inherit** from `nn.Module`, which provides parameter handling, device management, and autograd integration.  
- Understanding classes and inheritance makes deep learning frameworks more transparent and your own models easier to design, debug, and extend.
